# Exploratory Data Analysis — Credit Risk & Credit Limit Decisions

**Course:** Data Science & AI for Business  

**Context:** LendingClub accepted loans; we model **probability of default (PD)** and treat **`loan_amnt` as a proxy for credit limit** to support **prescriptive** decisions (adjust limits → PD changes → expected profit).

This notebook:
- Explores **drivers of default** (FICO, DTI, interest rate, loan amount, etc.)
- Emphasizes the **action variable** `loan_amnt` because changing it is the lever we will simulate
- Links patterns in the data to **why PD must be recomputed** when limits change

**Run location:** Use the **repository root** as the working directory (folder that contains `output/` and this `.ipynb`). The setup cell falls back to the parent directory if `output/` is not found.


## 0. Setup

Load libraries, resolve paths to `output/clean_data.csv` and `output/pd_predictions.csv` (same artifacts produced by the modeling pipeline). No new data is introduced.


In [ ]:
import os
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import metrics

# --- styling: readable plots for slides / report ---
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 13

# --- resolve project root (project/ contains output/) ---
_cwd = Path.cwd().resolve()
if (_cwd / "output" / "clean_data.csv").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "output" / "clean_data.csv").exists():
    PROJECT_ROOT = _cwd.parent
else:
    raise FileNotFoundError(
        "Could not find output/clean_data.csv. Run this notebook from the project root "
        "or from project/notebooks/ after generating outputs (preprocess + train_model)."
    )

DATA_PATH = PROJECT_ROOT / "output" / "clean_data.csv"
PD_PATH = PROJECT_ROOT / "output" / "pd_predictions.csv"

print("PROJECT_ROOT =", PROJECT_ROOT)


In [ ]:
# Load cleaned modeling table (features + default label)
df = pd.read_csv(DATA_PATH, low_memory=False)

# Optional: ensure default is integer for grouping / plots
df["default"] = df["default"].astype(int)

# PD file aligns row-for-row with df after pipeline (same row order)
pdf = pd.read_csv(PD_PATH, low_memory=False)
pdf["default"] = pdf["default"].astype(int)

print("clean_data:", df.shape)
print("pd_predictions:", pdf.shape)
display(df.head())


## 1. Data overview

Quick sanity check: we only observe **completed** loans (`Fully Paid` vs `Charged Off` in the raw pipeline), so `default` is a **realized outcome** suitable for learning PD. Later, when we **change** `loan_amnt` in simulation, we are asking: *what PD would our fitted model assign under a counterfactual limit?* — not replaying history verbatim.


In [ ]:
print(df.info())
print("\nDefault rate:", f"{df['default'].mean():.2%}")
print("loan_amnt describe:\n", df["loan_amnt"].describe())


## 2. Foundational risk driver: FICO (credit quality)

**Business meaning:** FICO summarizes repayment history and credit usage; lenders use it to price and approve risk.

**Credit risk link:** Lower FICO → higher historical stress → higher default frequency in many portfolios.

**Why it matters for limit decisions:** A borrower with weak FICO may generate **more PD sensitivity** to higher limits (stretching capacity). EDA here motivates **segment-aware** policies rather than one-size-fits-all limit increases.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="default", y="fico_range_low", ax=ax)
ax.set_xticklabels(["No default (0)", "Default (1)"])
ax.set_xlabel("Outcome")
ax.set_ylabel("FICO range (low)")
ax.set_title("FICO vs default — lower scores cluster with defaults")
plt.tight_layout()
plt.show()


**Read:** Defaulting borrowers tend to have **lower** FICO lows. For optimization, combining FICO segments with **limit changes** (via the model) helps avoid raising limits on already fragile credits without compensation through price or other controls.


## 3. Action variable — `loan_amnt` (proxy for credit limit)

**Business framing:** In this project, **`loan_amnt` is our proxy for the credit limit / exposure** we can adjust. In LendingClub data it is the **origination amount**; for decision-making we interpret *increasing the approved amount* as *increasing exposure* analogous to raising a line.

**Prescriptive point:** Default risk is **not fixed** in `loan_amnt`: larger balances can raise payment burden and loss given default exposure. EDA below asks whether, **in historical data**, larger amounts associate with higher default rates **holding our feature set as context** (the fitted model will later quantify PD shifts when we perturb `loan_amnt`).


In [ ]:
# --- Binned loan_amnt: 10 quantile bins (equal-count); duplicates='drop' handles ties ---
df["loan_amnt_bin"] = pd.qcut(df["loan_amnt"], q=10, duplicates="drop")

bin_stats = (
    df.groupby("loan_amnt_bin", observed=True)
    .agg(
        n=("default", "size"),
        default_rate=("default", "mean"),
        loan_median=("loan_amnt", "median"),
    )
    .reset_index()
)
bin_stats["default_rate_pct"] = bin_stats["default_rate"] * 100

fig, ax1 = plt.subplots(figsize=(10, 5))
x = range(len(bin_stats))
ax1.bar(x, bin_stats["default_rate_pct"], color="steelblue", alpha=0.85)
ax1.set_xticks(list(x))
ax1.set_xticklabels([str(iv) for iv in bin_stats["loan_amnt_bin"]], rotation=45, ha="right")
ax1.set_ylabel("Default rate (%)")
ax1.set_xlabel("Loan amount bin (quantiles)")
ax1.set_title("Default rate by loan amount decile (proxy for limit / exposure)")
plt.tight_layout()
plt.show()

display(bin_stats[["loan_amnt_bin", "n", "loan_median", "default_rate_pct"]])


**Interpretation:** Compare bar heights left → right (low to high amount bins). **If default rate rises with bin**, historical data suggests **higher exposure correlates with higher default frequency** — consistent with using `loan_amnt` as a lever that should **move PD** in model-based simulation. If the pattern is flat or non-monotonic, that is also informative: it pushes us to rely on the **fitted model’s partial effect** (and interactions encoded in other features) rather than a naive “bigger loans are always riskier” rule.

**Decision link:** This EDA motivates **re-scoring PD after limit changes** (your professor’s feedback): the relationship between amount and default is an empirical prior; the **logistic (or future) model** makes the counterfactual quantitative for each borrower.


In [ ]:
# Optional: smooth view — median loan in bin vs default rate
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bin_stats["loan_median"], bin_stats["default_rate_pct"], marker="o", color="darkred")
ax.set_xlabel("Median loan amount in bin ($)")
ax.set_ylabel("Default rate (%)")
ax.set_title("Default rate vs median loan amount by bin")
plt.tight_layout()
plt.show()


## 4. Pricing & capacity: interest rate and DTI

### 4.1 Interest rate vs default

**Business meaning:** Rate is the **price of risk** and reflects perceived credit quality at origination.

**Decision relevance:** When simulating limit changes, **pricing may need to move with risk**; EDA shows how realized defaults differ across rate levels in-sample.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="default", y="int_rate", ax=ax)
ax.set_xticklabels(["No default (0)", "Default (1)"])
ax.set_xlabel("Outcome")
ax.set_ylabel("Interest rate (%)")
ax.set_title("Interest rate vs default")
plt.tight_layout()
plt.show()


**Read:** Higher **coupon** often co-occurs with default in subprime-style segments (riskier borrowers get higher rates *and* higher default propensity). For limit optimization, **margin** (rate × balance) competes with **expected loss** (PD × LGD proxy); EDA motivates joint view of **amount + rate + PD** in profit logic.


### 4.2 Debt-to-income (DTI) vs default

**Business meaning:** DTI measures **repayment capacity** relative to income.

**Decision relevance:** Raising limits increases monthly burden if spend/utilization rises; borrowers with **already high DTI** may be closer to distress — an interaction to watch in modeling and simulation.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="default", y="dti", ax=ax)
ax.set_xticklabels(["No default (0)", "Default (1)"])
ax.set_xlabel("Outcome")
ax.set_ylabel("DTI")
ax.set_title("DTI vs default")
plt.tight_layout()
plt.show()

# Distribution overlay (optional complementary view)
fig, ax = plt.subplots(figsize=(8, 5))
for label, g in df.groupby("default"):
    g["dti"].hist(bins=40, alpha=0.45, ax=ax, label=f"default={label}", density=True)
ax.set_xlabel("DTI")
ax.set_ylabel("Density")
ax.set_title("DTI distribution by default")
ax.legend()
plt.tight_layout()
plt.show()


**Read:** Higher DTI mass among defaulters suggests **capacity constraints** matter. For **credit limit** decisions, combine this with `loan_amnt` shifts: simulation should surface when a limit increase pushes **expected profit** down because PD rises faster than revenue from a larger balance.


## 5. Other structural risk tags (grade, term, home ownership)

One-hot columns encode **underwriting grade**, **term**, and **home ownership**. We derive readable labels for plotting **default rate by category** (visualization only; underlying data unchanged).


In [ ]:
grade_cols = [c for c in df.columns if c.startswith("grade_")]


def row_grade(row):
    # Reference category A: all grade_* false
    for c in grade_cols:
        val = row[c]
        if val in (True, "True", 1, "1"):
            return c.replace("grade_", "")
    return "A"


df["grade_label"] = df.apply(row_grade, axis=1)

df["term_label"] = np.where(df["term_ 60 months"].astype(str).isin(["True", "1", True]), "60 months", "36 months")

hom_cols = [c for c in df.columns if c.startswith("home_ownership_")]


def row_home(row):
    for c in hom_cols:
        val = row[c]
        if val in (True, "True", 1, "1"):
            return c.replace("home_ownership_", "")
    return "OTHER/UNK"


df["home_label"] = df.apply(row_home, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

order_g = sorted(df["grade_label"].unique())
df.groupby("grade_label")["default"].mean().reindex(order_g).plot(kind="bar", ax=axes[0], color="teal")
axes[0].set_title("Default rate by grade")
axes[0].set_ylabel("Default rate")
axes[0].tick_params(axis="x", rotation=0)

df.groupby("term_label")["default"].mean().plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Default rate by term")
axes[1].set_ylabel("Default rate")

df.groupby("home_label")["default"].mean().sort_values(ascending=False).plot(kind="bar", ax=axes[2], color="slategray")
axes[2].set_title("Default rate by home ownership")
axes[2].set_ylabel("Default rate")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


**Business read:** Grades summarize **underwriter risk buckets**; longer **term** can raise cumulative default probability; **rent** sometimes correlates with higher default frequency in consumer credit (life-cycle / wealth proxies — interpret cautiously). These segments inform **who** gets limit changes in a prescriptive policy, not only **how much**.


## 6. Connecting to the PD model outputs

We already train a model that outputs **PD** for each loan. Here we **validate directionally** that PD aligns with realized outcomes (ROC) and visualize **PD vs `loan_amnt`**, the same feature we will perturb in optimization.


In [ ]:
cmap = {0: "green", 1: "red"}
fig, ax = plt.subplots(figsize=(8, 5))
for label, g in pdf.groupby("default"):
    ax.scatter(g["loan_amnt"], g["PD"], c=cmap[label], alpha=0.12, s=8, label=f"default={label}")
ax.set_xlabel("Loan amount ($)")
ax.set_ylabel("Predicted PD")
ax.set_title("Predicted PD vs loan amount (colored by realized default)")
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

fpr, tpr, _ = metrics.roc_curve(pdf["default"], pdf["PD"])
auc = metrics.roc_auc_score(pdf["default"], pdf["PD"])
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, label=f"ROC (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC — PD vs realized default")
ax.legend()
plt.tight_layout()
plt.show()
print("AUC:", auc)


**Read:** Scatter density shows whether **higher amounts** associate with **higher model-based PD** on average (again, a descriptive check; the **operational** use is counterfactual scoring). ROC/AUC summarizes **ranking ability** of PD for separating defaulters in this sample — useful for communication, while **profit optimization** will use **calibrated probabilities** in expectation formulas.


## 7. Implications for Credit Limit Optimization

### Why `loan_amnt` works as a **proxy for credit limit**
- In revolving products, the **line** caps outstanding balance; in installment data, **`loan_amnt` is the funded exposure** at origination. For prescriptive analytics we treat **“approving a higher amount”** as **“offering more credit capacity”** in a stylized way the professor suggested: **one model**, perturb a **dollar exposure** feature, read off **new PD**.

### Why changing `loan_amnt` should change **PD**
- **Direct channel:** payment burden and loss exposure scale with balance.  
- **Indirect channel (in the model):** `loan_amnt` enters the **same information set** as other risk drivers (FICO, DTI, rate, grade); shifting it changes the **linear predictor** in the logistic model → **PD moves**. That is exactly the **counterfactual** your instructor emphasized — not assuming PD is fixed after a limit change.

### How EDA supports the **simulation** step
- Binned default rates vs amount provide a **sanity check** that exposure and default co-move in history.  
- DTI / FICO / rate plots clarify **which customers** are fragile when limits increase.  
- The **next step** in the project pipeline: for each borrower, **vary `loan_amnt`**, **recompute PD** with the saved model + scaler, plug into **expected profit**, and search for a **better limit** (subject to business constraints you define in the report).

### Caveats (good to mention in class)
- Historical association **≠ causal**; simulation uses the **model as a scenario engine**, not a randomized experiment.  
- **Pricing, macro cycle, and selection** (who gets larger loans) confound simple EDA; still, the **process** matches prescriptive DS: **action → re-estimated risk → decision**.


## 8. Summary checklist (for the team)

| Item | Status in this notebook |
|------|-------------------------|
| Business-oriented framing of risk plots | ✓ Markdown under each section |
| **Action variable** `loan_amnt` — quantile bins & default rate | ✓ Section 3 |
| Interest rate & DTI vs default | ✓ Section 4 |
| Grade / term / home ownership default rates | ✓ Section 5 |
| Link to **PD outputs** (scatter + ROC) | ✓ Section 6 |
| **Credit limit optimization** narrative | ✓ Section 7 |

**Data note:** Inputs are only `output/clean_data.csv` and `output/pd_predictions.csv` from your existing pipeline — no new datasets.
